In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup, AutoConfig
from tqdm import tqdm

import optuna

import warnings
warnings.filterwarnings('ignore')

## EDA

In [ ]:
ner_data_path = Path('../data/ner/animals_ner_dataset.csv')
df = pd.read_csv(ner_data_path)
df.info()

In [ ]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.info()

In [ ]:
df.head()

In [ ]:
df['animal'].value_counts()

In [ ]:
df[df['animal'] == 'animal']

In [ ]:
df = df[df['animal'] != 'animal']
df['animal'].value_counts()

In [ ]:
# Split the dataset into train, validation, and test sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['animal'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['animal'])
print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

## NER label preparation

In [ ]:
# Function to create NER examples
def make_ner_example(sentence, animal):
    tokens = re.findall(r'\w+|[^\w\s]', sentence)
    labels = ['O'] * len(tokens)

    animal_lower = animal.lower()
    for idx, token in enumerate(tokens):
        if token.lower() == animal_lower:
            labels[idx] = 'B-ANIMAL'

    return tokens, labels


In [ ]:
tokens, labels = make_ner_example("The cat is on the roof.", "cat")
print("Tokens:", tokens)
print("Labels:", labels)

In [ ]:
train_df[['tokens', 'labels']] = train_df.apply(
    lambda row: make_ner_example(row['sentence'], 
    row['animal']), axis=1, result_type='expand'
)
val_df[['tokens', 'labels']] = val_df.apply(
    lambda row: make_ner_example(row['sentence'], 
    row['animal']), axis=1, result_type='expand'
)
test_df[['tokens', 'labels']] = test_df.apply(
    lambda row: make_ner_example(row['sentence'], 
    row['animal']), axis=1, result_type='expand'
)

print(train_df.head())

## Label encoding and model initialization

In [ ]:
# Create label mappings
label_to_id = {'O': 0, 'B-ANIMAL': 1}
id_to_label = {0: 'O', 1: 'B-ANIMAL'}

In [ ]:
# 
train_df['label_ids'] = train_df['labels'].apply(lambda labels: [label_to_id[label] for label in labels])
val_df['label_ids'] = val_df['labels'].apply(lambda labels: [label_to_id[label] for label in labels])
test_df['label_ids'] = test_df['labels'].apply(lambda labels: [label_to_id[label] for label in labels])

In [ ]:
train_df.head()

In [ ]:
model_name = 'bert-base-cased'
num_classes = len(label_to_id)

config = AutoConfig.from_pretrained(model_name, num_labels=num_classes)
tokenizer = AutoTokenizer.from_pretrained(model_name)

## Tokenization and label alignment

In [ ]:
max_length = 32

def tokenize_and_align_labels(tokens, label_ids):
    tokenized_inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    token_to_word = tokenized_inputs.word_ids()

    aligned_labels = []
    previous_word_id = None

    for word_id in token_to_word:
        if word_id is None:
            aligned_labels.append(-100)
        elif word_id != previous_word_id:
            aligned_labels.append(label_ids[word_id])
        else:
            aligned_labels.append(-100)

        previous_word_id = word_id

    return {
        "input_ids": tokenized_inputs["input_ids"],
        "attention_mask": tokenized_inputs["attention_mask"],
        "labels": aligned_labels
    }

In [ ]:
# Tokenize and align labels for the datasets
train_encoded = train_df.apply(
    lambda row: tokenize_and_align_labels(row['tokens'], 
    row['label_ids']), 
    axis=1
)

val_encoded = val_df.apply(
    lambda row: tokenize_and_align_labels(row['tokens'], 
    row['label_ids']), 
    axis=1
)

test_encoded = test_df.apply(
    lambda row: tokenize_and_align_labels(row['tokens'], 
    row['label_ids']), 
    axis=1
)

In [ ]:
train_encoded[0]

## NER Dataset and DataLoader

In [ ]:
class NERDataset(Dataset):
    def __init__(self, encoded_data):
        self.encoded_data = encoded_data.reset_index(drop=True)

    def __len__(self):
        return len(self.encoded_data)

    def __getitem__(self, idx):
        item = self.encoded_data.iloc[idx]

        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long)
        }

In [ ]:
# Create PyTorch datasets
train_dataset = NERDataset(train_encoded)
val_dataset = NERDataset(val_encoded)
test_dataset = NERDataset(test_encoded)

train_dataset[0]

In [ ]:
batch_size = 16
num_workers = 0
num_epochs = 3
learning_rate = 5e-5
weight_decay = 0.01

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=num_workers
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers
)

In [ ]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

## Model initialization

In [ ]:
def build_model(model_name, num_classes):
    config = AutoConfig.from_pretrained(model_name, num_labels=num_classes)
    model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)
    return model

In [ ]:
def create_optimizer(model, learning_rate, weight_decay):
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    return optimizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_model(model_name, num_classes).to(device)
optimizer = create_optimizer(model, learning_rate=2e-5, weight_decay=0.01)

### Train

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(dataloader)
    return avg_loss

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        preds = preds.cpu().numpy()
        labels = labels.cpu().numpy()

        for pred_seq, label_seq in zip(preds, labels):
            for pred, label in zip(pred_seq, label_seq):
                if label != -100:
                    all_preds.append(pred)
                    all_labels.append(label)

    return all_preds, all_labels

In [ ]:
best_val_f1 = 0.0
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}")

    val_preds, val_labels = evaluate(model, val_loader, device)
    val_accuracy = accuracy_score(val_labels, val_preds)
    val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(val_labels, val_preds, average='weighted')
    print(f"Validation Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1 Score: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        save_path = "../models/best_ner_model.pt'"
        torch.save(model.state_dict(), save_path)
        print("Best model saved!")

### Test

In [ ]:
model_load_path = "../models/ner_bert/best_ner_model.pt"
model.load_state_dict(torch.load(model_load_path))
model.to(device)

test_preds, test_labels = evaluate(model, test_loader, device)
test_accuracy = accuracy_score(test_labels, test_preds)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(test_labels, test_preds, average='weighted')

print(
    f"Test Accuracy: {test_accuracy:.4f}, "
    f"Precision: {test_precision:.4f}, "
    f"Recall: {test_recall:.4f}, "
    f"F1 Score: {test_f1:.4f}"
) 

In [ ]:
with open("../models/ner_bert/label_to_id.json", "w") as f:
    json.dump(label_to_id, f, indent=4)

id_to_label = {str(v): k for k, v in label_to_id.items()}
with open("../models/ner_bert/id_to_label.json", "w") as f:
    json.dump(id_to_label, f, indent=4)

tokenizer.save_pretrained("../models/ner_bert/tokenizer")